In [1]:
# imports
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

In [2]:
# loading training dataset
df = pd.read_csv("train_mlp_extended_final_version.csv")

print(df.shape)

df.head()

(4927, 6)


,quality_score,best_similarity,margin,label,true_identity,gallery_identity
0,18.756882,0.646138,0.435389,1,NaN,NaN
1,13.540867,0.247855,0.035649,0,NaN,NaN
2,15.522821,0.209527,0.010815,0,NaN,NaN
3,14.507830,0.207770,0.006632,0,NaN,NaN
4,15.120111,0.534458,0.336702,1,NaN,NaN


In [3]:
# features and labels
X = df[
    [
        "quality_score",
        "best_similarity",
        "margin",
    ]
]

y = df["label"]
print(df["label"].value_counts())
print(df["label"].value_counts(normalize=True))
print(X.shape)
print(y.shape)

label
1    2487
0    2440
Name: count, dtype: int64
label
1    0.50477
0    0.49523
Name: proportion, dtype: float64
(4927, 3)
(4927,)


In [4]:
# train validation split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(X_train.shape)
print(X_val.shape)

(3941, 3)
(986, 3)


In [5]:
# scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)

joblib.dump(
    scaler,
    "mlp_scaler_finalExp.pkl",
)

print("Scaler Saved")

Scaler Saved


In [6]:
# tensor conversion
X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32,
)

X_val_tensor = torch.tensor(
    X_val_scaled,
    dtype=torch.float32,
)

y_train_tensor = torch.tensor(
    y_train.values,
    dtype=torch.long,
)

y_val_tensor = torch.tensor(
    y_val.values,
    dtype=torch.long,
)

In [7]:
class MLPExperiment(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(3,32),
            nn.ReLU(),

            nn.Linear(32,16),
            nn.ReLU(),

            nn.Linear(16,8),
            nn.ReLU(),

            nn.Linear(8,2)
        )

    def forward(self,x):

        return self.network(x)

In [8]:
# model setup
model = MLPExperiment()

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0005,
    weight_decay=1e-4,
)

print(model)

MLPExperiment(
  (network): Sequential(
    (0): Linear(in_features=3, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=16, bias=True)
    (3): ReLU()
    (4): Linear(in_features=16, out_features=8, bias=True)
    (5): ReLU()
    (6): Linear(in_features=8, out_features=2, bias=True)
  )
)


In [9]:
# training the model
epochs = 300
best_epoch=0
best_val_accuracy = 0

for epoch in range(epochs):

    model.train()

    outputs = model(X_train_tensor)

    loss = criterion(
        outputs,
        y_train_tensor,
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    model.eval()

    with torch.no_grad():

        val_outputs = model(X_val_tensor)

        predictions = torch.argmax(
            val_outputs,
            dim=1,
        )

        val_accuracy = accuracy_score(
            y_val_tensor.numpy(),
            predictions.numpy(),
        )

    if val_accuracy >= best_val_accuracy:

        best_val_accuracy = val_accuracy
        best_epoch=epoch

        torch.save(
            model.state_dict(),
            "mlp_finalExp_model.pth",
        )

    if epoch % 10 == 0:

        print(
            f"Epoch {epoch:3d} | "
            f"Loss {loss.item():.4f} | "
            f"Val Acc {val_accuracy:.4f}"
        )

print()

print(
    "Best Validation Accuracy:",
    best_val_accuracy,
)

Epoch   0 | Loss 0.6871 | Val Acc 0.5325
Epoch  10 | Loss 0.6757 | Val Acc 0.5588
Epoch  20 | Loss 0.6632 | Val Acc 0.6156
Epoch  30 | Loss 0.6498 | Val Acc 0.8012
Epoch  40 | Loss 0.6354 | Val Acc 0.9037
Epoch  50 | Loss 0.6187 | Val Acc 0.9108
Epoch  60 | Loss 0.5996 | Val Acc 0.9128
Epoch  70 | Loss 0.5784 | Val Acc 0.9148
Epoch  80 | Loss 0.5553 | Val Acc 0.9168
Epoch  90 | Loss 0.5312 | Val Acc 0.9158
Epoch 100 | Loss 0.5079 | Val Acc 0.9138
Epoch 110 | Loss 0.4866 | Val Acc 0.9138
Epoch 120 | Loss 0.4675 | Val Acc 0.9158
Epoch 130 | Loss 0.4509 | Val Acc 0.9158
Epoch 140 | Loss 0.4358 | Val Acc 0.9209
Epoch 150 | Loss 0.4210 | Val Acc 0.9229
Epoch 160 | Loss 0.4046 | Val Acc 0.9270
Epoch 170 | Loss 0.3828 | Val Acc 0.9300
Epoch 180 | Loss 0.3578 | Val Acc 0.9290
Epoch 190 | Loss 0.3316 | Val Acc 0.9290
Epoch 200 | Loss 0.3046 | Val Acc 0.9290
Epoch 210 | Loss 0.2775 | Val Acc 0.9310
Epoch 220 | Loss 0.2512 | Val Acc 0.9290
Epoch 230 | Loss 0.2267 | Val Acc 0.9290
Epoch 240 | Loss

In [10]:
# loading the best model
best_model = MLPExperiment()

best_model.load_state_dict(
    torch.load(
        "mlp_finalExp_model.pth",
        map_location=torch.device("cpu"),
    )
)

best_model.eval()

print("Best Model Loaded")

Best Model Loaded


In [11]:
# validation metrics
with torch.no_grad():

    outputs = best_model(X_val_tensor)

    predictions = torch.argmax(
        outputs,
        dim=1,
    )

y_pred = predictions.numpy()

y_true = y_val_tensor.numpy()

In [12]:
# final metrics (acfuracy, precision, F1, Recall)
accuracy = accuracy_score(
    y_true,
    y_pred,
)

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0,
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0,
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0,
)

print("Accuracy :", accuracy)
print("Best validation accuracy:", best_val_accuracy)
print("Precision:", precision)
print("Best Epoch:", best_epoch)
print("Recall   :", recall)

print("F1 Score :", f1)

Accuracy : 0.9563894523326572
Best validation accuracy: 0.9563894523326572
Precision: 0.9809725158562368
Best Epoch: 299
Recall   : 0.9317269076305221
F1 Score : 0.9557157569515963


In [13]:
# confusion matrix
cm = confusion_matrix(
    y_true,
    y_pred,
)

print(cm)

print()

print(
    classification_report(
        y_true,
        y_pred,
    )
)

[[479   9]
 [ 34 464]]

              precision    recall  f1-score   support

           0       0.93      0.98      0.96       488
           1       0.98      0.93      0.96       498

    accuracy                           0.96       986
   macro avg       0.96      0.96      0.96       986
weighted avg       0.96      0.96      0.96       986



In [14]:
# TRR TAR FRR FAR
TN, FP, FN, TP = cm.ravel()

TAR = TP / (TP + FN)

FRR = FN / (TP + FN)

TRR = TN / (TN + FP)

FAR = FP / (TN + FP)

print()

print("TAR:", TAR)

print("FRR:", FRR)

print("TRR:", TRR)

print("FAR:", FAR)


TAR: 0.9317269076305221
FRR: 0.06827309236947791
TRR: 0.9815573770491803
FAR: 0.018442622950819672


In [15]:
train_df = pd.read_csv("train_mlp_extended_final_version.csv")

print(train_df[["quality_score","best_similarity","margin"]].describe())

       quality_score  best_similarity       margin
count    4927.000000      4927.000000  4927.000000
mean       20.477054         0.437326     0.187381
std         5.744799         0.209545     0.205331
min        12.476462        -0.008804    -0.417395
25%        16.051408         0.252436     0.017812
50%        19.161661         0.358792     0.087700
75%        23.504196         0.643469     0.382903
max        47.028496         0.892248     0.697794
